# UCS420: Cognitive Computing — Assignment 7 (NLP-I)

In [ ]:
!pip -q install gdown nbformat nltk pandas matplotlib

In [ ]:
import os, sys, re, subprocess, collections, math, string
import nbformat
from IPython import get_ipython
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

STARTER_ID = "1Xr9A49-kiRxA_HYf4v03XZsy3eBPrgcv"
STARTER_PATH = "/content/UCS420_LA9_Starter.ipynb"
download = subprocess.run([sys.executable, "-m", "gdown", "--id", STARTER_ID, "-O", STARTER_PATH],
                          capture_output=True, text=True)
if download.returncode != 0 or not os.path.exists(STARTER_PATH):
    print(download.stdout, download.stderr)
    raise RuntimeError("Automatic download failed. Open the starter link in the assignment PDF, download the .ipynb, upload it to Colab as /content/UCS420_LA9_Starter.ipynb, and rerun this cell.")
with open(STARTER_PATH, encoding="utf-8") as f:
    starter = nbformat.read(f, as_version=4)
ip = get_ipython()
for i, cell in enumerate(starter.cells):
    if cell.cell_type == "code" and cell.source.strip():
        print(f"Executing unchanged official starter code cell {i+1}...")
        result = ip.run_cell(cell.source, store_history=False)
        if result.error_before_exec or result.error_in_exec:
            raise RuntimeError(f"Official starter code cell {i+1} failed; see error above.")
ns = ip.user_ns
print("Starter executed. generate_dataset:", callable(ns.get("generate_dataset")),
      "| process_ticket:", callable(ns.get("process_ticket")))

## 1. Dataset and preprocessing

The original `process_ticket()` is called on every raw ticket. The summary reports sentence count, token count, punctuation tokens, stopwords, unique tokens, and final tokens. Fields are read from the function output when available; missing fields use transparent NLTK-based fallbacks.

In [ ]:
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag

# Find the generated raw ticket list in the official starter namespace.
ticket_var = None
for name in ["tickets", "raw_tickets", "ticket_texts", "dataset", "data"]:
    value = ns.get(name)
    if isinstance(value, (list, tuple)) and value and all(isinstance(x, str) for x in value):
        ticket_var = name
        break
if ticket_var is None:
    candidates = [(k, v) for k, v in ns.items()
                  if isinstance(v, (list, tuple)) and 12 <= len(v) <= 16 and all(isinstance(x, str) for x in v)]
    ticket_named = [(k, v) for k, v in candidates if "ticket" in k.lower()]
    if len(ticket_named) == 1: ticket_var = ticket_named[0][0]
    elif len(candidates) == 1: ticket_var = candidates[0][0]
if ticket_var is None:
    print("Candidate string lists:", [(k, len(v)) for k, v in ns.items()
          if isinstance(v, (list, tuple)) and v and all(isinstance(x, str) for x in v)])
    raise RuntimeError("Could not identify raw ticket variable. Set raw_tickets = <starter's ticket list> and rerun.")
raw_tickets = list(ns[ticket_var])
processed = [ns["process_ticket"](t) for t in raw_tickets]
print(f"Using {ticket_var}: {len(raw_tickets)} tickets.")
print("process_ticket() output example:", repr(processed[0])[:1800])

try: STOPWORDS = set(stopwords.words("english"))
except Exception: STOPWORDS = set("a an the and or but if of at by for with to from in on is am are was were be been being have has had do does did this that it its we you they he she".split())

def find_field(obj, wanted):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower().replace(" ", "_") in wanted: return v
        for v in obj.values():
            got = find_field(v, wanted)
            if got is not None: return got
    return None

def as_tokens(value):
    if value is None: return None
    if isinstance(value, str): return value.split()
    if isinstance(value, (list, tuple, set)): return [str(x) for x in value]
    return None

def fields_for(raw, result):
    toks = as_tokens(find_field(result, {"tokens", "all_tokens", "word_tokens", "tokenized"}))
    final = as_tokens(find_field(result, {"final_tokens", "clean_tokens", "processed_tokens", "filtered_tokens"}))
    punct = as_tokens(find_field(result, {"punctuation_tokens", "punct_tokens", "punctuation"}))
    stops = as_tokens(find_field(result, {"stopwords", "stop_words", "stopword_tokens"}))
    if toks is None:
        try: toks = word_tokenize(raw)
        except Exception: toks = re.findall(r"\\w+|[^\\w\\s]", raw)
    if punct is None: punct = [x for x in toks if x and all(c in string.punctuation for c in x)]
    if stops is None: stops = [x for x in toks if x.lower() in STOPWORDS]
    if final is None: final = [x for x in toks if x.lower() not in STOPWORDS and not (x and all(c in string.punctuation for c in x))]
    try: sentences = len(sent_tokenize(raw))
    except Exception: sentences = len([x for x in re.split(r"(?<=[.!?])\\s+", raw.strip()) if x])
    return {"sentences": sentences, "tokens": toks, "punctuation": punct, "stopwords": stops, "final": final}

ticket_fields = [fields_for(raw, out) for raw, out in zip(raw_tickets, processed)]
rows = []
for i, f in enumerate(ticket_fields, 1):
    rows.append({"Ticket": str(i), "Number of sentences": f["sentences"], "Number of tokens": len(f["tokens"]),
                 "Number of punctuation tokens": len(f["punctuation"]), "Number of stopwords": len(f["stopwords"]),
                 "Number of unique tokens": len(set(x.lower() for x in f["tokens"])), "Number of final tokens": len(f["final"])})
summary = pd.DataFrame(rows)
total = {"Ticket": "TOTAL"}
for c in summary.columns[1:]:
    total[c] = int(summary[c].sum()) if c != "Number of unique tokens" else "— (non-additive)"
display(pd.concat([summary, pd.DataFrame([total])], ignore_index=True))
selected = 1
f = ticket_fields[selected-1]
print(f"Selected ticket #{selected}:", raw_tickets[selected-1])
print("Actual tokens:", f["tokens"])
print("Punctuation tokens:", f["punctuation"])
print("Stopwords:", f["stopwords"])
print("Final tokens:", f["final"])

## 2. Regex extraction and replacement

Extraction uses `re.findall()` patterns for words longer than five letters, numbers/versions, capitalized words, and vowel-initial words. `re.sub()` replaces email addresses, URLs, and phone numbers with placeholders. The same patterns are tested on `TEST_LINES` and the generated tickets.

In [ ]:
TEST_LINES = ns.get("TEST_LINES", [
    "Please email help.desk@example.com about Wi-Fi version 2.0.1.",
    "The server reached 95.5% CPU; call +1 (555) 123-4567.",
    "Visit https://support.example.com/reset?mode=fast for state-of-the-art guidance.",
    "Isn't the VPN working? Contact Admin at admin@example.org."
])
print("TEST_LINES:")
for line in TEST_LINES: print("-", line)

long_pattern = re.compile(r"\b[A-Za-z]+(?:['’][A-Za-z]+)?\b")
number_pattern = re.compile(r"(?<![\w.])\d+(?:\.\d+)+(?![\w.])|(?<![\w.])\d+(?![\w.])")
capital_pattern = re.compile(r"\b[A-Z][a-zA-Z]*\b")
vowel_pattern = re.compile(r"\b[aeiouAEIOU][A-Za-z]*(?:['’][A-Za-z]+)?\b")

def regex_extract(text):
    words = long_pattern.findall(text)
    return {
        "words longer than 5 letters": [w for w in words if len(re.sub(r"['’]", "", w)) > 5],
        "numbers and versions": number_pattern.findall(text),
        "capitalized words": capital_pattern.findall(text),
        "words beginning with a vowel": vowel_pattern.findall(text)
    }

EMAIL_RE = re.compile(r"(?i)\b[A-Z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Z0-9-]+(?:\.[A-Z0-9-]+)+\b")
URL_RE = re.compile(r"(?i)\b(?:https?://|www\.)[^\s<>]+")
PHONE_RE = re.compile(r"(?<!\w)(?:\+?\d{1,3}[ .-]?)?(?:\(?\d{3}\)?[ .-]?)\d{3}[ .-]?\d{4}(?!\w)")
def redact(text):
    text = re.sub(EMAIL_RE, "<EMAIL>", text)
    text = re.sub(URL_RE, "<URL>", text)
    text = re.sub(PHONE_RE, "<PHONE>", text)
    return text

for label, lines in [("TEST_LINES", TEST_LINES), ("Generated tickets", raw_tickets)]:
    print("\\n---", label, "---")
    for i, line in enumerate(lines, 1):
        print(f"Line {i}: {line}")
        for k, v in regex_extract(line).items(): print(f"  {k}: {v}")
        print("  redacted:", redact(line))

## 3. Custom tokenization

`my_tokenize(text)` uses one `RegexpTokenizer` pattern that prioritizes URLs, email addresses, phone numbers, versions/decimals, contractions, and hyphenated terms. It runs on both `TEST_LINES` and all generated tickets, then compares results with `word_tokenize()`.

In [ ]:
CUSTOM_PATTERN = r"(?i)(?:https?://|www\.)[^\s<>]+|[A-Z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Z0-9-]+(?:\.[A-Z0-9-]+)+|(?:\+?\d{1,3}[ .-]?)?(?:\(?\d{3}\)?[ .-]?)\d{3}[ .-]?\d{4}|\d+(?:\.\d+)+|\d+|[A-Z]+(?:['’][A-Z]+)+|[A-Z]+(?:-[A-Z]+)+|[A-Z]+|[^\w\s]"
custom_tokenizer = RegexpTokenizer(CUSTOM_PATTERN, flags=re.IGNORECASE)
def my_tokenize(text):
    return custom_tokenizer.tokenize(text)

for label, lines in [("TEST_LINES", TEST_LINES), ("Generated tickets", raw_tickets)]:
    print("\\n---", label, "---")
    for line in lines:
        print("TEXT:", line)
        print("my_tokenize:", my_tokenize(line))
        try: print("word_tokenize:", word_tokenize(line))
        except Exception as e: print("word_tokenize error:", e)

print("\\nAt least three differences:")
print("1. Contractions: isn't is retained as one token by the custom pattern; word_tokenize commonly splits it into is and n't.")
print("2. Hyphenated terms: state-of-the-art is kept together by the custom pattern; word_tokenize usually splits at hyphens.")
print("3. Versions/decimals: 2.0.1 and 95.5 are retained as single tokens by the custom pattern; word_tokenize may split at periods.")
print("4. URLs/emails/phone numbers: the custom pattern prioritizes each structured string as one token; word_tokenize often splits it into components.")

## 4. Stemming and lemmatization

The comparison uses unique final tokens and displays only words whose Porter, Lancaster, RegexpStemmer, and POS-aware WordNet lemma outputs differ. The RegexpStemmer suffix is chosen from suffixes occurring in at least two sufficiently long tokens in this dataset when possible.

In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()
unique_final = sorted(set(str(t) for f in ticket_fields for t in f["final"] if str(t).strip()))
suffixes = ["ing", "ed", "ies", "es", "s", "ly", "ment", "tion"]
suffix_matches = {s: [w for w in unique_final if len(w) > len(s)+2 and w.lower().endswith(s)] for s in suffixes}
valid = [(s, ws) for s, ws in suffix_matches.items() if len(ws) >= 2]
if valid: regexp_suffix, support_words = max(valid, key=lambda x: (len(x[1]), -len(x[0])))
else: regexp_suffix, support_words = "ing", [w for w in unique_final if w.lower().endswith("ing")]
regexp_stemmer = RegexpStemmer(regexp_suffix + "$", min=3)
print(f"RegexpStemmer selected suffix: {regexp_suffix}")
print("Dataset words supporting suffix:", support_words)
if len(support_words) >= 2:
    print(f"Justification: this suffix occurs in at least two dataset words, e.g. {support_words[0]!r} and {support_words[1]!r}.")
else:
    print("No candidate suffix occurred in two sufficiently long tokens; 'ing' is used as a fallback.")

def wn_pos(tag):
    if tag.startswith("J"): return "a"
    if tag.startswith("V"): return "v"
    if tag.startswith("N"): return "n"
    if tag.startswith("R"): return "r"
    return "n"
tagged = pos_tag(unique_final) if unique_final else []
pos_map = {word: wn_pos(tag) for word, tag in tagged}
stem_rows = []
for word in unique_final:
    values = [porter.stem(word), lancaster.stem(word), regexp_stemmer.stem(word),
              lemmatizer.lemmatize(word, pos=pos_map.get(word, "n"))]
    if len(set(values)) > 1:
        stem_rows.append({"Original word": word, "Porter": values[0], "Lancaster": values[1],
                          f"Regexp ({regexp_suffix})": values[2], "WordNet lemma (POS-aware)": values[3]})
stem_df = pd.DataFrame(stem_rows, columns=["Original word", "Porter", "Lancaster", f"Regexp ({regexp_suffix})", "WordNet lemma (POS-aware)"])
print(f"Unique final tokens compared: {len(unique_final)}. Only differing outputs are shown.")
display(stem_df)

## 5. Corpus statistics

S1 = all tokens; S2 = tokens without punctuation; S3 = final tokens; S4 = S3 after Porter stemming. Report total tokens, unique tokens, and TTR. Also show the top ten words, S3 log(rank) versus log(frequency), and top five bigrams for S1 and S3.

In [ ]:
S1 = [str(t) for f in ticket_fields for t in f["tokens"]]
S2 = [t for t in S1 if not (t and all(ch in string.punctuation for ch in t))]
S3 = [str(t) for f in ticket_fields for t in f["final"]]
S4 = [porter.stem(t) for t in S3]

def corpus_stats(name, tokens):
    return {"Corpus": name, "Total tokens": len(tokens), "Unique tokens": len(set(t.lower() for t in tokens)),
            "TTR": len(set(t.lower() for t in tokens))/len(tokens) if tokens else 0}
stats = pd.DataFrame([corpus_stats("S1: all tokens", S1), corpus_stats("S2: without punctuation", S2),
                      corpus_stats("S3: final tokens", S3), corpus_stats("S4: S3 after Porter stemming", S4)])
display(stats.style.format({"TTR": "{:.4f}"}))

def top_words(tokens, n=10):
    return collections.Counter(t.lower() for t in tokens).most_common(n)
print("Top 10 words in S1:")
display(pd.DataFrame(top_words(S1, 10), columns=["Word", "Frequency"]))
print("Top 10 words in S3:")
display(pd.DataFrame(top_words(S3, 10), columns=["Word", "Frequency"]))

freqs = [count for _, count in collections.Counter(t.lower() for t in S3).most_common()]
if freqs:
    ranks = range(1, len(freqs)+1)
    plt.figure(figsize=(7, 4.5))
    plt.plot([math.log10(r) for r in ranks], [math.log10(f) for f in freqs], marker="o", linewidth=1)
    plt.xlabel("log10(rank)"); plt.ylabel("log10(frequency)")
    plt.title("S3: log(rank) vs log(frequency)"); plt.grid(True, alpha=0.3); plt.show()
else: print("S3 is empty; no rank-frequency plot can be drawn.")

def top_bigrams(tokens, n=5):
    normalized = [t.lower() for t in tokens]
    return collections.Counter(zip(normalized, normalized[1:])).most_common(n)
print("Top 5 bigrams in S1:")
display(pd.DataFrame([{"Bigram": f"{a} {b}", "Frequency": c} for (a,b), c in top_bigrams(S1, 5)]))
print("Top 5 bigrams in S3:")
display(pd.DataFrame([{"Bigram": f"{a} {b}", "Frequency": c} for (a,b), c in top_bigrams(S3, 5)]))

## 6. Interpretation (dataset-specific)

The next cell prints concise answers for (a) TTR change from S1 to S4, (b) a concrete tokenizer difference, (c) an observed example of potentially undesirable stemming, and (d) an example where POS-aware lemmatization differs from Porter stemming.

In [ ]:
lookup = {row["Corpus"]: row for _, row in stats.iterrows()}
a, b = lookup["S1: all tokens"], lookup["S4: S3 after Porter stemming"]
print("(a) TTR changes from S1 to S4 because preprocessing removes punctuation and stopwords, and Porter stemming can merge related forms into the same stem. In this dataset:")
print(f"    S1 TTR={a['TTR']:.4f} ({a['Unique tokens']} unique / {a['Total tokens']} total); S4 TTR={b['TTR']:.4f} ({b['Unique tokens']} unique / {b['Total tokens']} total).")

example = None
for line in TEST_LINES:
    mt = my_tokenize(line)
    try: wt = word_tokenize(line)
    except Exception: wt = []
    if mt != wt:
        example = (line, mt, wt); break
if example:
    print("\\n(b) Text:", example[0])
    print("    my_tokenize:", example[1])
    print("    word_tokenize:", example[2])
    print("    Explanation: the custom pattern intentionally preserves structured forms such as contractions, hyphenated terms, versions, and contact strings.")
else:
    print("\\n(b) Add isn't or 2.0.1 to TEST_LINES if no tokenizer difference appears.")

changed = [(w, porter.stem(w)) for w in unique_final if porter.stem(w) != w.lower()]
bad = [(w, s) for w, s in changed if len(w) >= 5 and (len(s) <= 3 or not w.lower().startswith(s[:2]))]
if not bad: bad = changed
if bad:
    w, s = bad[0]
    print(f"\\n(c) Potentially undesirable stemming example: {w!r} → {s!r}. Stemmers apply suffix rules without understanding meaning and can produce non-words or remove useful distinctions.")
else: print("\\n(c) No changed Porter stem found in this corpus; consult the stem-comparison table for any changed output.")

lemma_examples = []
for w in unique_final:
    lemma = lemmatizer.lemmatize(w, pos=pos_map.get(w, "n"))
    ps = porter.stem(w)
    if lemma != ps and lemma != w.lower():
        lemma_examples.append((w, lemma, ps, pos_map.get(w, "n")))
if lemma_examples:
    w, lemma, ps, pos = lemma_examples[0]
    print(f"\\n(d) POS-aware lemmatization example: {w!r} (POS={pos}) → lemma {lemma!r}; Porter stem is {ps!r}. POS-aware lemmatization uses a grammatical category and WordNet vocabulary, which can preserve a recognizable dictionary form.")
else:
    print("\\n(d) No nontrivial lemma-vs-Porter example was found in these tokens. The conceptual advantage is that POS-aware lemmatization returns dictionary forms, while stemming applies surface-form rules.")

print("\\nFinal checks: ticket count =", len(raw_tickets))
print("Original starter functions still available:",
      callable(ns.get("generate_dataset")), callable(ns.get("process_ticket")))
print("Retain the 8-character fingerprint printed by the official starter for submission.")